# **Shivprasad A. Mahind (123B1B266)**

# **1. Introduction to Support Vector Machines (SVM)**
Support Vector Machines (SVM) are a powerful and versatile class of supervised machine learning algorithms used for classification, regression, and outlier detection. For classification, the core idea of an SVM is to find the optimal hyperplane that best separates the data points of different classes in a high-dimensional space.

A hyperplane is a "decision boundary." In a 2D space, it's a line; in a 3D space, it's a plane. The SVM algorithm seeks the hyperplane that has the maximum margin—the largest distance between the hyperplane and the nearest data points from any class. These nearest points are called support vectors, as they are the critical elements that "support" or define the position of the hyperplane. By maximizing the margin, the SVM creates a robust decision boundary that is less likely to misclassify new, unseen data.

# **2. The Kernel Trick: Handling Non-Linear Data**
The MNIST digit dataset is complex and not linearly separable, meaning a simple straight line (or flat hyperplane) cannot effectively separate all 10 digit classes. This is where the kernel trick becomes essential.

The kernel trick allows SVMs to create complex, non-linear decision boundaries. It works by mapping the data into a higher-dimensional space where it is linearly separable, without the massive computational cost of actually transforming the data. The two kernels you tested are:

Linear Kernel ('linear'): This is the simplest kernel. It doesn't perform any transformation and attempts to find a linear hyperplane in the original feature space. It's fast but only works if the data is linearly separable. The main hyperparameter is C.

Radial Basis Function (RBF) Kernel ('rbf'): This is a powerful, non-linear kernel that can create highly complex decision boundaries. It's a popular default choice as it works well in many situations. The RBF kernel's effectiveness is controlled by two main hyperparameters: C and gamma.

# **3. Key SVM Hyperparameters**
Your GridSearchCV aims to find the optimal values for the following crucial hyperparameters:

**Regularization Parameter (C)**
The C parameter controls the trade-off between achieving a low training error (fitting the training data well) and a low testing error (generalizing to new data). It dictates the "cost" of misclassification.

**Low C:** A smaller C value creates a wider margin. The model is more tolerant of misclassified points on the training set, leading to a simpler decision boundary. This can reduce overfitting but might lead to underfitting (a soft margin).

**High C:** A larger C value creates a narrower margin. The model tries to classify every training example correctly, which can lead to a very complex decision boundary that is sensitive to noise and may overfit the training data (a hard margin).

**Gamma Parameter (gamma, γ)**
The gamma parameter is specific to the RBF kernel. It defines how much influence a single training example has. You can think of it as the "reach" of each support vector.

**Low gamma:** A small gamma means a single training example has a far reach. The decision boundary is smoother and less complex. This can lead to underfitting.

**High gamma:** A large gamma means a single training example has a very limited reach. The decision boundary becomes highly irregular and closely follows the training data, which can easily lead to overfitting.

In [ ]:
import numpy as np
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# --- 1. Load and Prepare the MNIST Dataset ---
print("1. Fetching and preparing MNIST dataset...")

mnist = fetch_openml('mnist_784', version=1, parser='auto')
X_full, y_full = mnist.data, mnist.target.astype(int)

X_full = X_full / 255.0

# --- 2. Create a Smaller Subset for Hyperparameter Tuning (Due to high computational cost of SVM) ---
sample_size = 10000
np.random.seed(42) # Set seed for reproducibility
sample_indices = np.random.choice(X_full.shape[0], sample_size, replace=False)

X_train_cv = X_full.iloc[sample_indices]
y_train_cv = y_full.iloc[sample_indices]
print(f"   -> Using a subset of {sample_size} samples for Cross-Validation/Tuning.")

# --- 3. Define the Pipeline and Parameter Grid for GridSearch ---
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

param_grid = [
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1, 10]  # Regularization parameter C for linear
    },
    {
        'svm__kernel': ['rbf'],
        'svm__C': [1, 10],       # Regularization parameter C for RBF
        'svm__gamma': [0.001, 0.01] # Kernel coefficient gamma for RBF
    }
]

# --- 4. Perform K-Fold Cross-Validation using GridSearchCV ---
n_folds = 3 # Use 3-Fold CV for faster tuning; 5 or 10 is standard for final report.

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=KFold(n_splits=n_folds, shuffle=True, random_state=42),
    scoring='accuracy',
    verbose=2,
    n_jobs=-1 # Use all available CPU cores
)

print(f"\n2. Starting {n_folds}-Fold Cross-Validation using GridSearchCV...")
start_time = time.time()
grid_search.fit(X_train_cv, y_train_cv)
end_time = time.time()

print(f"   -> Grid Search completed in {end_time - start_time:.2f} seconds.")

# --- 5. Report Best Model and Final Evaluation ---
print("\n3. Cross-Validation Results:")
print("Best Parameters found:", grid_search.best_params_)
print(f"Best CV Accuracy Score: {grid_search.best_score_:.4f}")

# --- 6. Final Model Evaluation on the Subset ---
# Retrieve the best model
best_svm_model = grid_search.best_estimator_

# Predict on the same subset (for a quick final report)
y_pred = best_svm_model.predict(X_train_cv)
final_accuracy = accuracy_score(y_train_cv, y_pred)

print(f"\n4. Final Model Report (Trained on {sample_size} Samples):")
print(f"Accuracy on Validation Subset: {final_accuracy:.4f}")
print("Classification Report:\n", classification_report(y_train_cv, y_pred, zero_division=0))

1. Fetching and preparing MNIST dataset...
   -> Using a subset of 10000 samples for Cross-Validation/Tuning.

2. Starting 3-Fold Cross-Validation using GridSearchCV...
Fitting 3 folds for each of 7 candidates, totalling 21 fits
   -> Grid Search completed in 306.60 seconds.

3. Cross-Validation Results:
Best Parameters found: {'svm__C': 10, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Best CV Accuracy Score: 0.9402

4. Final Model Report (Trained on 10000 Samples):
Accuracy on Validation Subset: 0.9987
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       983
           1       1.00      1.00      1.00      1152
           2       1.00      1.00      1.00       967
           3       1.00      1.00      1.00      1034
           4       1.00      1.00      1.00       906
           5       1.00      1.00      1.00       937
           6       1.00      1.00      1.00       961
           7       1.00      1.00   